# attoDRY transport analysis
This notebook is read-only with respect to hardware and the acquisition database. The loader returns accepted attempts only unless explicitly overridden for audit.

In [ ]:
from pathlib import Path
from attodry_control.analysis import load_analysis_rows, load_gate_leakage_rows
from attodry_control.publication import generate_publication_plots, load_gate_calibration

database = None  # Path('../data/run.sqlite')
run_id = None  # 'replace-with-run-id'
rows = () if database is None or run_id is None else load_analysis_rows(database, run_id)
rows[:6]

In [ ]:
# Supply a separately established RMS excitation current before computing R=Vxx/I.
current_a_rms = None  # independent value; do not infer from an unknown path
if current_a_rms is not None:
    resistance = [row.signed_resistance_ohm(current_a_rms) for row in rows if row.role.value == 'xx' and row.harmonic == 1]
    resistance[:5]

In [ ]:
# Standard publication suite. It remains disabled until all explicit inputs are set.
output_dir = None  # Path('../analysis_output/run_id')
total_series_resistance_ohm = None
gate_calibration_path = None  # Path('../config/gate_calibration.local.toml')
if rows and output_dir is not None:
    leakage = load_gate_leakage_rows(database, run_id)
    calibration = None if gate_calibration_path is None else load_gate_calibration(gate_calibration_path)
    manifest = generate_publication_plots(
        rows, leakage, output_dir,
        total_series_resistance_ohm=total_series_resistance_ohm,
        gate_calibration=calibration,
    )
    manifest['manifest']